# JailbreakBench Reproduction Notebook

Interactive walkthrough of our scoped DSC 291 reproduction. Each section calls the same workflow functions used by the CLI scripts in `scripts/`.

**Google Colab users:** use [`colab_driver.ipynb`](colab_driver.ipynb) instead. Colab's default Python 3.12 kernel cannot import this project directly.

**Requirements:** Python 3.10 or 3.11 kernel, Linux/CUDA for Vicuna generation (A100 recommended), Hugging Face access for Vicuna + Llama-Guard-2, and an OpenAI API key for the GPT-4o-mini comparison.

For a non-interactive run, see [`README.md`](../README.md).

In [ ]:
from pathlib import Path

import pandas as pd

from jbb_repro.env import load_env_file
from jbb_repro.vllm_workflows import run_benign_vicuna, run_harmful_vicuna
from jbb_repro.workflows import (
    extract_qualitative_examples,
    run_openai_on_prompts,
    score_heuristic,
    score_llamaguard,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "jbb_repro").exists() and (REPO_ROOT.parent / "src" / "jbb_repro").exists():
    REPO_ROOT = REPO_ROOT.parent

CONFIGS = REPO_ROOT / "configs"
OUTPUTS = REPO_ROOT / "outputs"
REPORTS = REPO_ROOT / "reports"

load_env_file()
pd.set_option("display.max_colwidth", 120)

print(f"Repo root: {REPO_ROOT}")

## Setup

Run once per runtime:

```bash
pip install -e ".[dev,vllm]"
```

Set secrets in `/content/.env` on Colab or in a project-root `.env` / shell exports:

- `OPENAI_API_KEY`
- `HF_TOKEN`

## Step 0 — Verify wiring (dry run)

Build the sampled behavior subset and PAIR/GCG prompts without loading Vicuna.

In [ ]:
dry_run = run_harmful_vicuna(
    CONFIGS / "vicuna_vllm.yaml",
    limit=4,
    dry_run=True,
)
dry_run

## Step 1 — Vicuna-13B harmful attacks (PAIR + GCG)

Skip or reduce scope during smoke testing by setting `LIMIT = 2`. For the full reproduction, leave `LIMIT = None`.

In [ ]:
LIMIT = None  # set to a small integer for smoke tests

harmful_run = run_harmful_vicuna(CONFIGS / "vicuna_vllm.yaml", limit=LIMIT)
harmful_run

## Step 2 — Score Vicuna harmful responses

Primary judge: Llama-Guard-2. Heuristic scoring is included as a sensitivity check.

In [ ]:
harmful_llamaguard = score_llamaguard(harmful_run.response_path)
harmful_heuristic = score_heuristic(harmful_run.response_path)

display(harmful_llamaguard.summary)
display(harmful_heuristic.summary)

## Step 3 — Vicuna-13B benign behaviors

In [ ]:
benign_run = run_benign_vicuna(CONFIGS / "vicuna_benign_vllm.yaml", limit=LIMIT)
benign_heuristic = score_heuristic(benign_run.response_path, benign=True)
benign_llamaguard = score_llamaguard(benign_run.response_path)

display(benign_heuristic.refusal_summary)
display(benign_llamaguard.summary)

## Step 4 — Dictionary-filter defense

In [ ]:
defense_run = run_harmful_vicuna(
    CONFIGS / "vicuna_dictionary_filter_vllm.yaml",
    limit=LIMIT,
    defense="dictionary_filter",
)
defense_llamaguard = score_llamaguard(defense_run.response_path)
display(defense_llamaguard.summary)

## Step 5 — GPT-4o-mini on the same attack prompts

In [ ]:
api_run = run_openai_on_prompts(
    CONFIGS / "gpt4o_mini_jbb.yaml",
    harmful_run.prompt_path,
    limit=LIMIT,
)
api_llamaguard = score_llamaguard(api_run.response_path)
api_heuristic = score_heuristic(api_run.response_path)

display(api_llamaguard.summary)
display(api_heuristic.summary)

## Step 6 — Qualitative examples and paper comparison

Paper Vicuna harmful ASR: PAIR 69%, GCG 80%. Our scoped reproduction targets similar numbers on the Llama-Guard-2 judge.

In [ ]:
extract_qualitative_examples(
    harmful_llamaguard.scored_path,
    REPORTS / "qualitative_examples_vicuna.md",
)
extract_qualitative_examples(
    api_llamaguard.scored_path,
    REPORTS / "qualitative_examples_gpt4o_mini.md",
)

paper_comparison = pd.DataFrame(
    [
        {"attack": "PAIR", "paper_vicuna_asr": 0.69, "our_vicuna_asr": None},
        {"attack": "GCG", "paper_vicuna_asr": 0.80, "our_vicuna_asr": None},
    ]
)
for method in ("PAIR", "GCG"):
    row = harmful_llamaguard.summary[harmful_llamaguard.summary["method"] == method]
    if not row.empty:
        paper_comparison.loc[paper_comparison["attack"] == method, "our_vicuna_asr"] = float(
            row.iloc[0]["attack_success_rate"]
        )

display(paper_comparison)
print("Reports written under reports/qualitative_examples_*.md")